# v34 Advocate rerank and full selector

CPU-only. Reads `v34_base_advocate_candidates.json` generated by a GPU account and tests strict rerank rules. Selects only if it beats v32.2 and passes artifact verification.

In [ ]:
import json, re, math, os, sys, random, itertools, statistics
from pathlib import Path
from collections import Counter, defaultdict

CANDIDATE_DIRS = [
    Path('/kaggle/working'),
    Path('/kaggle/input/datasets/yixuanisthebest/v2333333'),
    Path('/kaggle/input/datasets/nguyenminhtric/test-pipeline'),
    Path('/kaggle/input'),
    Path('/mnt/data'),
]

def find_file(names, required=True):
    if isinstance(names, str): names = [names]
    for d in CANDIDATE_DIRS:
        for name in names:
            p = d / name
            if p.exists(): return p
    for d in CANDIDATE_DIRS:
        if d.exists():
            for root, dirs, files in os.walk(d):
                for name in names:
                    if name in files:
                        return Path(root)/name
    if required: raise FileNotFoundError(f'Could not find any of {names} in {CANDIDATE_DIRS}')
    return None

def load_json(names, required=True):
    p = find_file(names, required=required)
    if p is None: return None, None
    with open(p, 'r', encoding='utf-8') as f: return json.load(f), p

def save_json(obj, name):
    p = Path('/kaggle/working')/name
    try: p.parent.mkdir(parents=True, exist_ok=True)
    except Exception: pass
    with open(p, 'w', encoding='utf-8') as f: json.dump(obj, f, ensure_ascii=False, indent=2)
    print('saved', p)
    return p

LABELS = ['A','B','C','D','Yes','No','Unknown']

def safe_pred(x):
    return x if x in LABELS else 'UNPARSEABLE'

def prf_for_label(rows, lab):
    tp=sum(1 for r in rows if r.get('gold')==lab and r.get('pred')==lab)
    fp=sum(1 for r in rows if r.get('gold')!=lab and r.get('pred')==lab)
    fn=sum(1 for r in rows if r.get('gold')==lab and r.get('pred')!=lab)
    prec=tp/(tp+fp) if tp+fp else 0.0
    rec=tp/(tp+fn) if tp+fn else 0.0
    f1=2*prec*rec/(prec+rec) if prec+rec else 0.0
    return {'precision':prec,'recall':rec,'f1':f1,'support':tp+fn,'pred_count':tp+fp}

def metrics(rows):
    n=len(rows); acc=sum(1 for r in rows if r.get('gold')==r.get('pred'))/n
    per={lab: prf_for_label(rows, lab) for lab in LABELS}
    macro=sum(per[lab]['f1'] for lab in LABELS)/len(LABELS)
    weighted=sum(per[lab]['f1']*per[lab]['support'] for lab in LABELS)/sum(per[lab]['support'] for lab in LABELS)
    mc=[r for r in rows if r.get('is_mc')]
    ynu=[r for r in rows if not r.get('is_mc')]
    mc_labs=['A','B','C','D']; ynu_labs=['Yes','No','Unknown']
    mc_macro=sum(prf_for_label(mc, lab)['f1'] for lab in mc_labs)/4 if mc else 0
    ynu_macro=sum(prf_for_label(ynu, lab)['f1'] for lab in ynu_labs)/3 if ynu else 0
    return {'n':n,'acc':acc,'macro_f1':macro,'weighted_f1':weighted,'mc_macro':mc_macro,'ynu_macro':ynu_macro,'per_label':per,
            'gold_count':dict(Counter(r.get('gold') for r in rows)), 'pred_count':dict(Counter(r.get('pred') for r in rows))}

def diffs(a,b):
    return [r1['idx'] for r1,r2 in zip(a,b) if r1.get('pred')!=r2.get('pred')]

def clone_rows(rows): return [dict(r) for r in rows]

def load_baselines():
    v27, p27 = load_json('v27_standard_preds.json', required=False)
    base, pb = load_json(['v32_2_full_preds.json','v32_2_independent_full_preds.json','v32_2_standard_preds.json','v30_1_full_preds.json'], required=True)
    print('baseline path:', pb)
    base_m = metrics(base)
    print('baseline macro:', base_m['macro_f1'])
    return v27, p27, base, pb, base_m

BANKED = {14,71,109,125}
BASELINE_MACRO = 0.596858548901435

In [ ]:
v27,p27,base,pb,base_m=load_baselines()
adv, ap = load_json('v34_base_advocate_candidates.json', required=True)
print('adv path', ap, 'meta', adv.get('__meta__'))

In [ ]:
def valid(a):
    return isinstance(a,dict) and a.get('verdict')=='VALID' and a.get('cited')

def invalid(a):
    return isinstance(a,dict) and a.get('verdict')=='INVALID'

variants=[]
# Variant definitions: conservative to permissive.
for name, cond in [
    ('S1_unique_valid_yes_no_unknown_invalid', lambda pack: valid(pack.get('Yes')) and invalid(pack.get('No')) and invalid(pack.get('Unknown'))),
    ('S2_yes_valid_no_invalid', lambda pack: valid(pack.get('Yes')) and invalid(pack.get('No'))),
    ('S3_yes_valid_no_not_valid_unknown_not_valid', lambda pack: valid(pack.get('Yes')) and not valid(pack.get('No')) and not valid(pack.get('Unknown'))),
]:
    flips=[]
    for r in base:
        if r.get('idx') in BANKED or r.get('is_mc') or r.get('pred')!='No': continue
        pack=adv.get(str(r['idx']), {})
        if cond(pack): flips.append(r['idx'])
    cand=clone_rows(base)
    for r in cand:
        if r['idx'] in set(flips):
            r['pred']='Yes'; r['repair']='v34_base_advocate_'+name
    m=metrics(cand)
    variants.append({'name':name,'flips':flips,'n_flips':len(flips),'metrics':m,'delta_macro':m['macro_f1']-base_m['macro_f1'],
                     'posthoc_gold':dict(Counter(r['gold'] for r in base if r['idx'] in set(flips)))})

variants=sorted(variants, key=lambda x:x['metrics']['macro_f1'], reverse=True)
for v in variants:
    print(v['name'], 'macro', v['metrics']['macro_f1'], 'delta', v['delta_macro'], 'n', v['n_flips'], 'gold', v['posthoc_gold'], 'flips', v['flips'])
save_json({'version':'v34_base_advocate_rerank_leaderboard','baseline_v32_2':base_m,'variants':variants}, 'v34_base_advocate_rerank_leaderboard.json')

In [ ]:
selected=None
if variants:
    top=variants[0]
    m=top['metrics']; per=m['per_label']; bper=base_m['per_label']
    collapse = per['Yes']['pred_count']>45 or per['No']['pred_count']<30 or per['No']['f1'] < bper['No']['f1']-0.03 or per['Unknown']['f1'] < bper['Unknown']['f1']
    if top['delta_macro'] > 0.01 and m['mc_macro']==base_m['mc_macro'] and not collapse:
        selected=top
if selected:
    out=clone_rows(base)
    for r in out:
        if r['idx'] in set(selected['flips']): r['pred']='Yes'; r['repair']='v34_base_advocate_selected_'+selected['name']
    save_json(out, 'v34_base_advocate_full_preds.json')
    reload,_=load_json('v34_base_advocate_full_preds.json')
    rm=metrics(reload)
    assert abs(rm['macro_f1']-selected['metrics']['macro_f1'])<1e-12
    summary={'version':'v34_base_advocate_full','selection_status':'SELECT_V34_BASE_ADVOCATE','selected':selected,'metrics':rm,'baseline_v32_2':base_m,
             'flipped_indices':selected['flips']}
    save_json(summary, 'v34_base_advocate_full_summary.json')
    print('SELECT BASE ADVOCATE', rm['macro_f1'])
else:
    save_json(base, 'v34_base_advocate_full_preds.json')
    summary={'version':'v34_base_advocate_full','selection_status':'ROLLBACK_V32_2','reason':'no advocate variant passed strict gates','metrics':base_m,'baseline_v32_2':base_m,
             'best_candidate':variants[0] if variants else None}
    save_json(summary, 'v34_base_advocate_full_summary.json')
    print('ROLLBACK BASE ADVOCATE')